# Homework04 - Task 5A Graph Preparation for Node Classification

This notebook follows the final-project protocol exactly:

1. Build a 3D CellComplex.
2. Assign a string `room_type` to each room cell.
3. Assign a string `door_type` to each aperture face.
4. Add apertures using `Topology.AddApertures(..., exclusive=False, subTopologyType="Face", tolerance=0.001)`.
5. Build graph using `Graph.ByTopology(..., direct=False, directApertures=True)`.
6. Export CSV files for node classification.

## Cell 1 - Imports

In [ ]:
from pathlib import Path

from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Topology import Topology
from topologicpy.Graph import Graph
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper

In [ ]:
print("TopologicPy version:", Helper.Version())

## Cell 2 - Allowed Labels (String Values)

In [ ]:
ALLOWED_DOOR_TYPES = {"passage", "door", "entrance door"}

# Edit this list to match your real project categories.
ALLOWED_ROOM_TYPES = {
    "livingroom",
    "bedroom",
    "kitchen",
    "bathroom",
    "corridor",
    "stair",
    "entrance",
    "service",
    "balcony",
    "storage"
}

def assert_valid_room_type(value: str):
    if not isinstance(value, str):
        raise TypeError(f"room_type must be string, got {type(value).__name__}")
    if value not in ALLOWED_ROOM_TYPES:
        raise ValueError(f"Invalid room_type '{value}'. Allowed: {sorted(ALLOWED_ROOM_TYPES)}")

def assert_valid_door_type(value: str):
    if not isinstance(value, str):
        raise TypeError(f"door_type must be string, got {type(value).__name__}")
    if value not in ALLOWED_DOOR_TYPES:
        raise ValueError(f"Invalid door_type '{value}'. Allowed: {sorted(ALLOWED_DOOR_TYPES)}")

print("Allowed door_type values:", sorted(ALLOWED_DOOR_TYPES))
print("Allowed room_type values:", sorted(ALLOWED_ROOM_TYPES))

## Cell 3 - Minimal Protocol Example (Same Structure as the Announcement)

In [ ]:
cc = CellComplex.Prism(wSides=1, vSides=1)
cells = Topology.Cells(cc)

d = Dictionary.ByKeyValue("room_type", "livingroom")
cells[0] = Topology.SetDictionary(cells[0], d)

d = Dictionary.ByKeyValue("room_type", "bedroom")
cells[1] = Topology.SetDictionary(cells[1], d)

f = Face.Rectangle(width=0.5, length=0.5, direction=[1, 0, 0])
d = Dictionary.ByKeyValue("door_type", "door")
f = Topology.SetDictionary(f, d)

cc = Topology.AddApertures(
    cc,
    [f],
    exclusive=False,
    subTopologyType="Face",
    tolerance=0.001
)

graph = Graph.ByTopology(cc, direct=False, directApertures=True)
print("Graph created.")
print("Nodes:", len(Graph.Vertices(graph)))
print("Edges:", len(Graph.Edges(graph)))

## Cell 4 - Validate Dictionaries on Cells, Apertures, and Graph Nodes

In [ ]:
for i, c in enumerate(Topology.Cells(cc)):
    d = Topology.Dictionary(c)
    room_type = Dictionary.ValueAtKey(d, "room_type")
    assert_valid_room_type(room_type)
    print(f"Cell {i}: room_type={room_type}")

apertures = Topology.Apertures(cc)
print("Apertures found:", len(apertures))
for i, a in enumerate(apertures):
    ad = Topology.Dictionary(a)
    door_type = Dictionary.ValueAtKey(ad, "door_type")
    assert_valid_door_type(door_type)
    print(f"Aperture {i}: door_type={door_type}")

for i, v in enumerate(Graph.Vertices(graph)):
    vd = Topology.Dictionary(v)
    room_type = Dictionary.ValueAtKey(vd, "room_type")
    if room_type is None:
        raise ValueError(f"Graph node {i} has no room_type.")
    assert_valid_room_type(room_type)
print("Node dictionary validation passed.")

## Cell 5 - Real Project Wiring (Homework04 Objects)

This section uses your real object files from `Homework04/Objects` to prepare a protocol-compliant graph:

1. Load room solids and assign `room_type` strings.
2. Load door and entrance-door faces and assign `door_type` strings.
3. Add apertures to the room CellComplex with the exact required settings.
4. Build graph with `direct=False` and `directApertures=True`.

If any room OBJ imports as shells/faces instead of cells, a shell-to-cell fallback is attempted.

In [ ]:
BASE = Path(r"C:\Users\etmaglari\IAAC\etmaglari_gML")
OBJECTS_DIR = BASE / "Homework04" / "Objects"

ROOM_FILES = {
    "livingroom": "Living room.obj",
    "bedroom": "Bedroom.obj",
    "kitchen": "Kitchen.obj",
    "bathroom": "Bathroom.obj",
    "corridor": "Corridor.obj",
    "stair": "Stair.obj"
}

DOOR_FILES = {
    "door": "door.obj",
    "entrance door": "Entrance door.obj"
}

def as_list(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]

def import_topology(path: Path):
    if not path.exists():
        return None
    top = Topology.ByOBJPath(str(path), selfMerge=False)
    tops = as_list(top)
    return tops[0] if tops else None

def topology_to_cells(top):
    if top is None:
        return []
    cells = as_list(Topology.Cells(top))
    if cells:
        return cells
    out = []
    for shell in as_list(Topology.Shells(top)):
        try:
            c = Cell.ByShell(shell, planarize=True)
            if c is not None:
                out.append(c)
        except Exception:
            pass
    return out

def set_room_type(cell, room_type):
    assert_valid_room_type(room_type)
    d = Dictionary.ByKeyValue("room_type", room_type)
    return Topology.SetDictionary(cell, d)

def aperture_faces_with_type(path: Path, door_type: str):
    assert_valid_door_type(door_type)
    top = import_topology(path)
    faces = as_list(Topology.Faces(top))
    tagged = []
    for f in faces:
        d = Dictionary.ByKeyValue("door_type", door_type)
        tagged.append(Topology.SetDictionary(f, d))
    return tagged

room_cells_real = []
for room_type, fname in ROOM_FILES.items():
    src = OBJECTS_DIR / fname
    top = import_topology(src)
    cells = topology_to_cells(top)
    for c in cells:
        room_cells_real.append(set_room_type(c, room_type))
    print(f"{fname}: cells={len(cells)}")

if not room_cells_real:
    raise ValueError("No room cells were created from Homework04/Objects. Check OBJ content.")

cc_real = CellComplex.ByCells(room_cells_real, transferDictionaries=True, silent=True)

apertures_real = []
for dtype, fname in DOOR_FILES.items():
    src = OBJECTS_DIR / fname
    tagged_faces = aperture_faces_with_type(src, dtype)
    apertures_real.extend(tagged_faces)
    print(f"{fname}: aperture_faces={len(tagged_faces)}")

# Optional manual passage apertures: append faces tagged with door_type='passage' here.
# passages = []
passages = []
for p in passages:
    d = Dictionary.ByKeyValue("door_type", "passage")
    apertures_real.append(Topology.SetDictionary(p, d))

cc_real = Topology.AddApertures(
    cc_real,
    apertures_real,
    exclusive=False,
    subTopologyType="Face",
    tolerance=0.001
)

graph_real = Graph.ByTopology(cc_real, direct=False, directApertures=True)

ACTIVE_CC = cc_real
ACTIVE_GRAPH = graph_real

print("Real CellComplex cells:", len(as_list(Topology.Cells(ACTIVE_CC))))
print("Apertures tagged:", len(apertures_real))
print("Real graph nodes:", len(Graph.Vertices(ACTIVE_GRAPH)))
print("Real graph edges:", len(Graph.Edges(ACTIVE_GRAPH)))

## Cell 6 - Export CSV for Node Classification

This export uses `ACTIVE_GRAPH` when available (real project wiring).
If not available, it falls back to the minimal example `graph`.

In [ ]:
BASE = Path(r"C:\Users\etmaglari\IAAC\etmaglari_gML")
EXPORT_DIR = BASE / "Homework04" / "Notebooks" / "dataset_node_classification_custom"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Keep nodeFeaturesKeys focused on features already available in node dictionaries.
# Add more keys after you enrich your cell dictionaries before graph creation.
node_features = ["room_type"]

graph_to_export = ACTIVE_GRAPH if "ACTIVE_GRAPH" in globals() else graph
source_name = "ACTIVE_GRAPH (real model)" if "ACTIVE_GRAPH" in globals() else "graph (minimal example)"

status = Graph.ExportToCSV(
    graph_to_export,
    path=str(EXPORT_DIR),
    nodeFeaturesKeys=node_features,
    overwrite=True
)

print("Graph source:", source_name)
print("Export status:", status)
print("Exported to:", EXPORT_DIR)

## Cell 7 - Verify Required CSV Files

In [ ]:
required = ["graphs.csv", "nodes.csv", "edges.csv"]
for name in required:
    p = EXPORT_DIR / name
    print(name, "->", "OK" if p.exists() else "MISSING", p)

## Cell 8 - Next Step

Use the exported folder directly in:
- `05 Homework04_etm_Node_Classification.ipynb` for training workflow
- `05B Homework04_etm_Pretrained_Inference.ipynb` for pretrained inference

Set dataset path there to:
`C:/Users/etmaglari/IAAC/etmaglari_gML/Homework04/Notebooks/dataset_node_classification_custom`